# Week 03 - Image Formation, Optics and Camera Calibration

**MCTE 4323 / MCTA 4364 Machine Vision**

### Learning objectives
By the end of this lab you will be able to:
- Describe the **pinhole camera model** and the intrinsic matrix $K$.
- Project 3D world points to 2D image points using `cv2.projectPoints`.
- Explain radial and tangential **lens distortion** and correct it with `cv2.undistort`.
- Recover camera intrinsics from chessboard views with `cv2.calibrateCamera`.

### The pinhole model
A point $(X, Y, Z)$ in the camera frame projects to pixel $(u, v)$:

$$
s\begin{bmatrix} u \\ v \\ 1 \end{bmatrix}
= K \begin{bmatrix} R & t \end{bmatrix}
\begin{bmatrix} X \\ Y \\ Z \\ 1 \end{bmatrix},
\qquad
K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}
$$

$f_x, f_y$ are focal lengths in pixels and $(c_x, c_y)$ is the principal point (usually the image centre).

## 1. Setup

In [ ]:
import os
if not os.path.isdir("MCTA-4364-Machine-Vision"):
    !git clone https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git
%cd MCTA-4364-Machine-Vision
!pip -q install opencv-python numpy matplotlib

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def show(*images, titles=None, cmap=None):
    titles = titles or [""] * len(images)
    plt.figure(figsize=(5 * len(images), 5))
    for i, img in enumerate(images):
        plt.subplot(1, len(images), i + 1)
        plt.imshow(img, cmap=cmap or (None if img.ndim == 3 else "gray"))
        plt.title(titles[i]); plt.axis("off")
    plt.tight_layout(); plt.show()

print("OpenCV:", cv2.__version__)

## 2. Guided example - projecting a 3D point
We define a camera `K` and project a 3D point in front of the camera. Note how $u, v$ scale with $f$ and are offset by the principal point.

In [ ]:
w, h = 640, 480
fx, fy = 800.0, 800.0
cx, cy = w / 2, h / 2
K = np.array([[fx, 0, cx],
              [0, fy, cy],
              [0,  0,  1]], dtype=np.float64)

point_3d = np.array([[[0.5, 0.0, 3.0]]], dtype=np.float64)  # (X, Y, Z) in metres
img_pt, _ = cv2.projectPoints(point_3d, np.zeros(3), np.zeros(3), K, None)
print("3D point:", point_3d.ravel())
print("Projected pixel:", img_pt.ravel())

## 3. Guided example - a synthetic chessboard
Calibration needs a target with known geometry. A chessboard works because the inner corner positions are known exactly.

In [ ]:
sq, cols, rows = 60, 10, 7          # 10x7 squares -> 9x6 inner corners
board = np.ones((rows * sq, cols * sq), np.uint8) * 255
for r in range(rows):
    for c in range(cols):
        if (r + c) % 2 == 0:
            board[r*sq:(r+1)*sq, c*sq:(c+1)*sq] = 0

pattern = (9, 6)
found, corners = cv2.findChessboardCorners(board, pattern)
print("Corners found:", found, "| number of corners:", len(corners))

vis = cv2.cvtColor(board, cv2.COLOR_GRAY2BGR)
cv2.drawChessboardCorners(vis, pattern, corners, found)
show(vis, titles=["Detected inner corners"])

## 4. Guided example - recovering the intrinsics
We now **simulate** several views of the board at different poses, project them with a known `K_true`, then let `calibrateCamera` recover `K` from scratch. This is the same algorithm used with a real camera.

In [ ]:
# 3D coordinates of the inner corners on the plane Z = 0
objp = np.zeros((pattern[0] * pattern[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:pattern[0], 0:pattern[1]].T.reshape(-1, 2)

K_true = K.copy()
dist_true = np.zeros(5)

# Six different board poses (rotation vectors + translations)
rvecs_true = [np.array([0.1, 0.0, 0.0]), np.array([0.2, 0.1, -0.1]),
              np.array([-0.1, 0.3, 0.2]), np.array([0.3, -0.2, 0.1]),
              np.array([0.0, 0.0, 0.4]), np.array([0.15, 0.15, 0.0])]
tvecs_true = [np.array([0, 0, 12.]), np.array([1, 1, 13.]),
              np.array([-1, 0.5, 11.]), np.array([0.5, -1, 14.]),
              np.array([0, 0, 15.]), np.array([1, -0.5, 13.])]

objpoints, imgpoints = [], []
for rvec, tvec in zip(rvecs_true, tvecs_true):
    imgp, _ = cv2.projectPoints(objp, rvec, tvec, K_true, dist_true)
    objpoints.append(objp)
    imgpoints.append(imgp)

ret, K_est, dist_est, _, _ = cv2.calibrateCamera(objpoints, imgpoints, (w, h), None, None)
print("RMS reprojection error: %.4f px" % ret)
print("\nK (true):\n", np.round(K_true, 1))
print("\nK (estimated):\n", np.round(K_est, 1))
print("\nFocal length error: %.3f px" % abs(K_true[0,0] - K_est[0,0]))

## 5. Guided example - lens distortion
Real lenses bend light, so straight lines appear curved. The dominant term is **radial distortion**:

$$x_{dist} = x(1 + k_1 r^2 + k_2 r^4), \qquad y_{dist} = y(1 + k_1 r^2 + k_2 r^4)$$

- $k_1 < 0$: **barrel** distortion
- $k_1 > 0$: **pincushion** distortion

Let's distort a grid and then undistort it.

In [ ]:
# Build a clean grid to make distortion obvious
grid = np.ones((h, w, 3), np.uint8) * 255
for x in range(0, w, 40):
    cv2.line(grid, (x, 0), (x, h), (0, 0, 0), 1)
for y in range(0, h, 40):
    cv2.line(grid, (0, y), (w, y), (0, 0, 0), 1)

K_g = K.copy()
dist_barrel = np.array([-0.35, 0.10, 0, 0, 0], np.float64)
dist_pincushion = np.array([0.35, 0.10, 0, 0, 0], np.float64)

barrel = cv2.undistort(grid, K_g, dist_barrel)
pincushion = cv2.undistort(grid, K_g, dist_pincushion)
show(grid, barrel, pincushion, titles=["Ideal grid", "Barrel (k1<0)", "Pincushion (k1>0)"])

> Note: `undistort` applies *negative* of the distortion model. If your result is the opposite of expected, flip the sign of $k_1$.

## 6. Exercise (complete the code)

1. Add **Gaussian noise** to the projected image points in Section 4 (use `np.random.normal(0, 0.5, imgp.shape)`).
2. Re-run calibration and report the RMS error and focal-length error.
3. Does the error grow? Why?

*This mirrors real calibration, where corner detection is never perfect.*

In [ ]:
# TODO: add noise to imgpoints, recalibrate, compare


## 7. Challenge (independent)

Undistort a real photograph. Load `resources/images/test_image.jpeg`, then call `cv2.undistort` with `dist = [-0.3, 0.1, 0, 0, 0]` and compare before/after. Try to find a value of $k_1$ that makes the image look "sunken" (barrel) versus "bulged" (pincushion).

In [ ]:
# Your code here


## 8. Reflection
1. What physical quantity does $f_x$ (pixels) represent, and why does it differ from the lens focal length in mm?
2. Why do we need at least three chessboard views to recover all intrinsics?
3. A robot arm must measure object positions in millimetres. Why is calibration essential before any measurement?

## 9. Visual summary

In [ ]:
import sys
sys.path.append("resources/scripts")
from cvhelpers import concept_map

concept_map([
    "3D world point (X, Y, Z)",
    "Extrinsics [R | t]: world -> camera frame",
    "Intrinsics K: camera -> image (focal length, principal point)",
    "Lens distortion (radial + tangential)",
    "Pixel (u, v)",
    "Calibration recovers K and distortion from known targets"
], title="The camera imaging pipeline")

## 10. Interactive exploration - distortion model

Slide `k1` from barrel (negative) through zero to pincushion (positive). What happens at `k1 = 0`?

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact

def distort_demo(k1=0.0):
    grid = np.ones((480, 640, 3), np.uint8) * 255
    for x in range(0, 640, 40):
        cv2.line(grid, (x, 0), (x, 480), (0, 0, 0), 1)
    for y in range(0, 480, 40):
        cv2.line(grid, (0, y), (640, y), (0, 0, 0), 1)
    out = cv2.undistort(grid, K, np.array([k1, 0, 0, 0, 0], np.float64))
    show(out, titles=[f"undistort with k1 = {k1:.2f}"])

interact(distort_demo, k1=widgets.FloatSlider(min=-0.6, max=0.6, step=0.05, value=0.0))

## 11. Check your understanding (Q&A)

<details><summary><b>Q1. What is the difference between intrinsic and extrinsic parameters?</b></summary>

Intrinsics (K, distortion) describe the camera itself and do not change when it moves. Extrinsics (R, t) describe the camera pose relative to the world.

</details>

<details><summary><b>Q2. Why does a robot arm that measures in millimetres depend on calibration?</b></summary>

Without calibration we cannot convert pixels to metric distances. Calibration links the image to real-world units through K and the known target geometry.

</details>

<details><summary><b>Q3. Why does adding noise to corner locations increase the calibration error?</b></summary>

Calibration solves a least-squares system over many correspondences. Noise perturbs the inputs, so the estimated parameters drift from the true ones.

</details>

## 12. Further reading & self-exploration
- OpenCV camera calibration: https://docs.opencv.org/4.x/dc/dbb/tutorial_py_calibration.html
- OpenCV camera model and distortion: https://docs.opencv.org/4.x/d9/d0c/group__calib3d.html
- Szeliski, *Computer Vision* (free) - Camera models chapter: https://szeliski.org/Book/
- Wikipedia - Pinhole camera model: https://en.wikipedia.org/wiki/Pinhole_camera_model
- Wikipedia - Distortion (optics): https://en.wikipedia.org/wiki/Distortion_(optics)

**Try next:** print a chessboard, capture 10-15 photos, and calibrate your phone camera.

## 13. Key takeaways
- The pinhole model maps 3D points to pixels via K, R, t.
- Calibration recovers intrinsics and distortion from a known target.
- Radial distortion causes barrel (k1<0) or pincushion (k1>0) bending.
- Calibration is a prerequisite for any metric measurement.
